# Push a trained FR5 checkpoint → HuggingFace Hub

Standalone — run in a **fresh kernel on the training pod** (needs the pod's torch to
read the checkpoint; everything else is derived from the checkpoint file itself, so
no training-notebook state is required). Works for any π-family checkpoint dir
(`checkpoints_pi0_v2`, `checkpoints_pi05_v2`, …).

What it does: **validates** the checkpoint (loads it, checks the keys deploy.py
needs), generates a **model card from the checkpoint's own config** + metrics, and
uploads to a versioned repo — safe to re-run any time (idempotent; re-pushing after
more training just updates the files).

## 1 · Parameters

In [ ]:
import os

HF_TOKEN   = os.environ.get("HF_TOKEN", "")
CKPT_DIR   = "/workspace/checkpoints_pi0_v2"   # which run to push
MODEL_REPO = "auto"                            # "auto" -> <you>/fr5-<policy>-v2, or set explicitly
DATASET_REPO = "Slifold/fr5-pick-place-lerobot-v2"   # named in the model card
PUSH_RESUME  = True                            # include last.pt (~13 GB: weights + optimizer).
                                               # False -> deploy-only push (best.pt + metrics)
PRIVATE      = True

if not HF_TOKEN:
    import getpass
    HF_TOKEN = getpass.getpass("HF token (hf_...): ").strip()
assert HF_TOKEN.startswith("hf_")
print("parameters set")

## 2 · Dependencies + login

In [ ]:
import importlib, subprocess, sys

subprocess.run([sys.executable, "-m", "pip", "install", "huggingface_hub>=0.34"], check=True)
try:
    import torch
except ImportError:
    raise SystemExit("torch not found — run this on the training pod (torch is needed "
                     "to read and validate the checkpoint)")
from huggingface_hub import login, whoami
login(token=HF_TOKEN)
print("logged in as:", whoami()["name"])

## 3 · Validate the checkpoint & extract its metadata

Loads `best.pt` once (a few minutes for ~8 GB off the network volume) and checks the
exact keys `deploy.py` rebuilds from — a corrupt or half-written checkpoint fails
HERE, not after a multi-GB upload. The model card below is generated from what the
checkpoint actually contains, not from hand-typed claims.

In [ ]:
import gc, pathlib, torch

ck_dir = pathlib.Path(CKPT_DIR)
best = ck_dir / "best.pt"
assert best.exists(), f"no best.pt in {ck_dir} — finish at least one epoch first"

print("files in", ck_dir)
for f in sorted(ck_dir.iterdir()):
    if f.is_file():
        print(f"   {f.name:22s} {f.stat().st_size/1e9:6.2f} GB")

print("\nloading best.pt to validate (~minutes on a network volume)...")
ck = torch.load(best, map_location="cpu", weights_only=False)
REQUIRED = {"model_state", "config", "stats", "policy", "epoch", "val_l1"}
missing = REQUIRED - set(ck)
assert not missing, f"checkpoint missing keys {missing} — deploy.py could not load this"

POLICY   = ck["policy"]
EPOCH    = int(ck["epoch"])
VAL_L1   = float(ck["val_l1"])
CFG      = ck["config"]
N_TENSORS = len(ck["model_state"])
ACTION_SPACE = ck.get("action_space", "joint")
m, d, t = CFG["model"], CFG["dataset"], CFG["training"]
print(f"\nvalid: policy={POLICY}  epoch={EPOCH}  val_l1={VAL_L1:.4f}  "
      f"tensors={N_TENSORS}  action_space={ACTION_SPACE}")
del ck
gc.collect()

## 4 · Model card (from the checkpoint's own config)

In [ ]:
import csv

best_val, best_epoch = VAL_L1, EPOCH
mp = ck_dir / "metrics.csv"
if mp.exists():
    rows = list(csv.DictReader(open(mp)))
    if rows:
        b = min(rows, key=lambda r: float(r["val_l1"]))
        best_val, best_epoch = float(b["val_l1"]), int(b["epoch"])

lora_rank    = m.get("vlm_lora_rank", 0)
lora_targets = m.get("vlm_lora_targets", ["q_proj", "k_proj", "v_proj", "o_proj"])
repo_name    = MODEL_REPO if MODEL_REPO != "auto" else f"{whoami()['name']}/fr5-{POLICY}-v2"

card = f"""---
license: apache-2.0
tags: [robotics, vla, {POLICY}, lerobot, fairino-fr5]
---
# {repo_name.split('/')[-1]} — {POLICY} finetuned on the FR5 multi-task dataset

- **Base**: `{m.get('pretrained', '?')}` · dtype {m.get('dtype', '?')} · quantize {m.get('quantize', 'none')}
- **Finetune**: LoRA r={lora_rank} on {len(lora_targets)} VLM Linear types ({', '.join(lora_targets)}) + fully-trained action expert
- **Data**: `{DATASET_REPO}` — chunk {d.get('chunk_size')} @ 30 Hz, cameras {d.get('camera_names')}, aug `{d.get('aug_level')}`, proprio `{m.get('proprio_mode')}`
- **Training**: batch {t.get('batch_size')}, lr {t.get('lr')} -> {t.get('lr_min', '?')} (warmup {t.get('warmup_steps', '?')}), max_steps {t.get('max_steps')}, action space `{ACTION_SPACE}`
- **Best val L1**: {best_val:.4f} (epoch {best_epoch})

## Deploy ([fairino-fr5-policies](https://github.com/SreevaatsavB/fairino-fr5-policies))
```bash
python common/deploy.py --hf-repo {repo_name}                    # pulls best.pt
python common/deploy.py --hf-repo {repo_name} --quantize nf4     # low-VRAM GPU (Turing+)
python common/deploy.py --hf-repo {repo_name} --n-action-steps 10  # receding horizon
```
`best.pt` carries model_state + config + normalization stats; `deploy.py` rebuilds the
exact wrapper from it. `last.pt` (if present) adds optimizer state for resume.
"""
print(card)

## 5 · Push

In [ ]:
from huggingface_hub import HfApi

repo = MODEL_REPO if MODEL_REPO != "auto" else f"{whoami()['name']}/fr5-{POLICY}-v2"
patterns = ["best.pt", "metrics.csv", "metrics_steps.csv"] + (["last.pt"] if PUSH_RESUME else [])
files = [f for f in patterns if (ck_dir / f).exists()]
total = sum((ck_dir / f).stat().st_size for f in files) / 1e9
print(f"pushing {files}  (~{total:.1f} GB) -> {repo}")

api = HfApi()
api.create_repo(repo, private=PRIVATE, exist_ok=True)
api.upload_file(path_or_fileobj=card.encode(), path_in_repo="README.md",
                repo_id=repo, commit_message="model card")
api.upload_folder(folder_path=str(ck_dir), repo_id=repo, allow_patterns=files,
                  commit_message=f"{POLICY} epoch {EPOCH} · val_l1 {VAL_L1:.4f}")

pushed = set(api.list_repo_files(repo))
assert "best.pt" in pushed and "README.md" in pushed
print(f"\nverified on the Hub: {sorted(pushed)}")
print(f"done -> https://huggingface.co/{repo}")

## Done

Re-run this notebook any time — after epoch 2 finishes, after a longer 30k-step run
pointed at the same `CKPT_DIR`, or for a different policy by changing `CKPT_DIR`
(the repo name follows the checkpoint's own `policy` field when `MODEL_REPO="auto"`).